# Capstone Projects and the Python Interview Bank

Notebook for **Chapter 14** of the *python* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/python/ch14.html

Tip: change a number, predict the output, then run the cell again.

## 14.1 Capstone 1: A Command-Line Tool With Logging, Tests and a Typed API

In [ ]:
import csv, io, statistics, math
from collections import defaultdict
data = """placed_at_s,prep_seconds,dish,served_ok
1000,120,salad,true
1100,95,salad,true
1200,2300,pasta,false
1500,180,salad,true
1800,450,pasta,true
2000,510,pasta,true
2400,130,soup,true
2900,700,soup,true
"""
def pct(xs, p):
    xs = sorted(xs); return xs[max(0, math.ceil(len(xs)*p/100)-1)]
by = defaultdict(list); err = defaultdict(int); stamps = []
for r in csv.DictReader(io.StringIO(data)):
    by[r["dish"]].append(int(r["prep_seconds"]))
    err[r["dish"]] += r["served_ok"] != "true"
    stamps.append(int(r["placed_at_s"]))
print(f"{'dish':8}{'n':>3}{'avg':>8}{'p90':>7}{'fail%':>7}")
for k, v in sorted(by.items()):
    print(f"{k:8}{len(v):>3}{statistics.mean(v):>8.1f}{pct(v,90):>7}{100*err[k]/len(v):>7.1f}")
n = sum(len(v) for v in by.values())
print("orders/min:", round(n / ((max(stamps)-min(stamps))/60), 2))

dish      n     avg    p90  fail%
pasta     3  1086.7   2300   33.3
salad     3   131.7    180    0.0
soup      2   415.0    700    0.0
orders/min: 0.25


## 14.2 Capstone 2: A FastAPI Service With a Database and Tests

In [ ]:
from fastapi.testclient import TestClient
from kitchen_api.main import create_app

def test_create_and_get_shift():
    client = TestClient(create_app(db_url=":memory:"))
    r = client.post("/v1/shifts", json={"name": "lunch", "started_at": "2026-01-31T10:00:00Z"})
    assert r.status_code == 201
    shift_id = r.json()["id"]
    assert client.get(f"/v1/shifts/{shift_id}").json()["name"] == "lunch"

def test_unknown_shift_is_404():
    client = TestClient(create_app(db_url=":memory:"))
    assert client.get("/v1/shifts/999").status_code == 404

In [ ]:
import sqlite3

class ShiftRepository:
    def __init__(self, path=":memory:"):
        self.con = sqlite3.connect(path)
        self.con.row_factory = sqlite3.Row
        self.con.execute("CREATE TABLE IF NOT EXISTS shifts("
                         "id INTEGER PRIMARY KEY, name TEXT NOT NULL)")
    def create(self, name):
        with self.con:
            cur = self.con.execute("INSERT INTO shifts(name) VALUES (?)", (name,))
        return cur.lastrowid
    def get(self, shift_id):
        row = self.con.execute("SELECT id, name FROM shifts WHERE id = ?", (shift_id,)).fetchone()
        return dict(row) if row else None
    def list(self, limit=10, offset=0):
        rows = self.con.execute("SELECT id, name FROM shifts ORDER BY id LIMIT ? OFFSET ?",
                                (limit, offset)).fetchall()
        return [dict(r) for r in rows]

repo = ShiftRepository()
for n in ("lunch", "dinner", "brunch"):
    print("created", repo.create(n))
print(repo.get(2))
print(repo.get(99))
print(repo.list(limit=2, offset=2))
try:
    repo.create(None)
except sqlite3.IntegrityError as e:
    print("IntegrityError:", e)

## 14.3 Capstone 3: A Queue and Service-Time Analyser

In [ ]:
prep = [120, 95, 2300, 180, 450, 510, 130, 700]
avg_min = sum(prep) / len(prep) / 60
lam = 8 / (1900 / 60)
print(sum(prep), round(avg_min, 2), round(lam, 4), round(lam * avg_min, 2), round(sum(prep) / 60 / (1900 / 60), 2))

4485 9.34 0.2526 2.36 2.36


## 14.4 Capstone 4: A Responsible Scraper

In [ ]:
from urllib.robotparser import RobotFileParser

ROBOTS = """User-agent: *
Disallow: /private/
Crawl-delay: 2
"""
rp = RobotFileParser(); rp.parse(ROBOTS.splitlines())
cache = {"https://site.test/b": "<html>cached b</html>"}
UA = "StudyBot/1.0 (me@example.com)"

def plan(url):
    if not rp.can_fetch(UA, url):
        return "SKIP (robots)"
    if url in cache:
        return "CACHE"
    return "FETCH (wait %ss first)" % rp.crawl_delay(UA)

for u in ["https://site.test/a", "https://site.test/b", "https://site.test/private/x"]:
    print(u, "->", plan(u))

https://site.test/a -> FETCH (wait 2s first)
https://site.test/b -> CACHE
https://site.test/private/x -> SKIP (robots)


## 14.5 The Mastery Tests and the 150-Problem Plan

In [ ]:
from collections import Counter, deque
import heapq

def two_sum(nums, target):
    seen = {}
    for i, x in enumerate(nums):
        if target - x in seen:
            return [seen[target - x], i]
        seen[x] = i

def valid_parens(s):
    pairs = {")": "(", "]": "[", "}": "{"}
    st = []
    for ch in s:
        if ch in "([{": st.append(ch)
        elif not st or st.pop() != pairs[ch]: return False
    return not st

def top_k(words, k):
    c = Counter(words)
    return [w for w, _ in sorted(c.items(), key=lambda kv: (-kv[1], kv[0]))[:k]]

def merge_intervals(iv):
    out = []
    for s, e in sorted(iv):
        if out and s <= out[-1][1]: out[-1][1] = max(out[-1][1], e)
        else: out.append([s, e])
    return out

def bfs_shortest(grid, start, goal):
    q = deque([(start, 0)]); seen = {start}
    while q:
        (r, c), d = q.popleft()
        if (r, c) == goal: return d
        for dr, dc in ((1,0),(-1,0),(0,1),(0,-1)):
            nr, nc = r+dr, c+dc
            if 0 <= nr < len(grid) and 0 <= nc < len(grid[0]) and grid[nr][nc] == 0 and (nr,nc) not in seen:
                seen.add((nr,nc)); q.append(((nr,nc), d+1))
    return -1

def anagram_groups(ws):
    g = {}
    for w in ws: g.setdefault("".join(sorted(w)), []).append(w)
    return list(g.values())

print(two_sum([2, 7, 11, 15], 9))
print(valid_parens("{[()]}"), valid_parens("([)]"))
print(top_k("a b a c b a d".split(), 2))
print(merge_intervals([[1,3],[8,10],[2,6],[15,18],[17,20]]))
grid = [[0,0,0],[1,1,0],[0,0,0]]
print(bfs_shortest(grid, (0,0), (2,0)))
print(anagram_groups(["eat","tea","tan","ate","nat","bat"]))
print(heapq.nlargest(3, [5, 1, 9, 3, 7]))

[0, 1]
True False
['a', 'b']
[[1, 6], [8, 10], [15, 20]]
6
[['eat', 'tea', 'ate'], ['tan', 'nat'], ['bat']]
[9, 7, 5]


## 14.6 The Python Interview Bank (45 Graded Questions)

## 14.7 Three More Interview Rounds, Worked in Full

The bank in 14.6 gives you many short questions. Real interviews also test how you think aloud. This section walks three rounds the way a strong candidate would: restate, predict, code, test, then name the trade-off.

In [ ]:
def make_counter():
    count = 0
    def inc():
        nonlocal count
        count += 1
        return count
    return inc

a, b = make_counter(), make_counter()
print(a(), a(), b())
fns = [lambda: i for i in range(3)]
print([f() for f in fns])
fixed = [lambda i=i: i for i in range(3)]
print([f() for f in fixed])

1 2 1
[2, 2, 2]
[0, 1, 2]


In [ ]:
from collections import OrderedDict

class LRUCache:
    """Least-recently-used cache with O(1) get and put."""
    def __init__(self, capacity):
        self.capacity = capacity
        self._data = OrderedDict()
    def get(self, key, default=None):
        if key not in self._data:
            return default
        self._data.move_to_end(key)
        return self._data[key]
    def put(self, key, value):
        self._data[key] = value
        self._data.move_to_end(key)
        if len(self._data) > self.capacity:
            self._data.popitem(last=False)
    def keys(self):
        return list(self._data)

c = LRUCache(2)
c.put("a", 1); c.put("b", 2)
c.get("a")
c.put("c", 3)
print(c.keys(), c.get("b"))
c.put("a", 10)
print(c.keys(), c.get("a"))
assert LRUCache(1).get("x", "none") == "none"

['a', 'c'] None
['c', 'a'] 10


In [ ]:
def append_item(x, bucket=[]):
    bucket.append(x)
    return bucket
print(append_item(1), append_item(2))

def append_safe(x, bucket=None):
    if bucket is None:
        bucket = []
    bucket.append(x)
    return bucket
print(append_safe(1), append_safe(2))

[1, 2] [1, 2]
[1] [2]


## Chapter recap

## Practice set

Problem 14.P.1 (easy) For the 8 orders in 14.1.1, what is the p90 of `salad` and why?

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/python/ch14.html